# 30 · CXR-LT — baseline **EHR-only**, 40 nhãn (Stage 2)

Đo xem **chỉ riêng EHR** đoán được bao nhiêu, chưa nhìn ảnh một lần nào. Đây là một trong ba mốc
gốc của kế hoạch (EHR-only · CXR-only · EHR+CXR).

## Ba tier trả lời ba câu tách bạch

```
Q_CTX   boi canh + missingness + dem do luong + do moi + age/gender
        KHONG mot gia tri lam sang nao
Q_CLIN  Q_CTX + gia tri labs (24h, 7d) + vitals + comorbidity
        KHONG mot cot vi sinh nao
Q_FULL  Q_CLIN + vi sinh + one-hot organism
```

Ý nghĩa nằm ở **hiệu**, không phải ở từng con số:

| hiệu | nói lên |
|---|---|
| `Q_CTX` cao một mình | EHR đang đọc ra **loại bệnh nhân** và **cường độ chăm sóc**, không đọc ra bệnh. Bệnh nhân ICU được đo 40 lần/ngày thì chính con số 40 đã nói họ nặng |
| **`Q_CLIN − Q_CTX`** | **giá trị lâm sàng thật thêm được bao nhiêu** — đây là con số đáng quan tâm nhất. Nếu ≈ 0 thì toàn bộ tín hiệu chỉ là phân tầng bối cảnh |
| `Q_FULL − Q_CLIN` | vi sinh thêm được bao nhiêu |

## Ba điều bắt buộc khi báo cáo

**Kèm `n_pos` cạnh mọi AP.** 11 nhãn tail chỉ có 129–903 ca dương trên train, và 4 nhãn có **dưới
50 ca dương trên val** — AP ở đó phương sai rất lớn, đọc trần trụi là tự lừa mình.

**Tách theo tier head/medium/tail.** Mất cân bằng 667× nên macro gộp che gần hết thông tin.

**Tách theo `setting`.** Nhóm `outpatient` (17.18%) **không có vitals** — MIMIC-IV không có bảng
vital ngoại trú nào. Nếu AP tụt hẳn khi chấm riêng trong từng nhóm `setting`, phần lớn tín hiệu
chỉ là phân tầng bối cảnh chứ không phải chẩn đoán.

**Kèm shuffle control.** Xáo nhãn trên train rồi khớp lại: nếu AP không sụt về mức prevalence thì
có rò rỉ ở đâu đó.

**CPU, ~30–60 phút** (40 nhãn × 3 tier + shuffle). Có `RESUME` từ CSV — đứt thì chạy lại tiếp.
Cần `29b_cxrlt_ehr_features.ipynb` đã chạy.

## C0 · Config + nạp dữ liệu

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore', category=UserWarning)

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'

OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
RES_DIR = os.path.join(OUT_DIR, 'stage2_ehr_only'); os.makedirs(RES_DIR, exist_ok=True)
P_EHR   = os.path.join(OUT_DIR, 'cxrlt_ehr.parquet')
P_SPEC  = os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')
P_INV   = os.path.join(OUT_DIR, 'cxrlt_inventory.json')
P_IDX   = os.path.join(OUT_DIR, 'cxrlt_index.parquet')
for p in (P_EHR, P_SPEC, P_INV, P_IDX):
    assert os.path.exists(p), f'thieu {p} -> chay 28 va 29b truoc'

QUICK  = False    # True -> chi 6 nhan, de thu duong ong truoc khi chay that
RESUME = True     # doc lai ket qua da co trong CSV, chi khop phan con thieu
SEED   = 42

SPEC = json.load(open(P_SPEC)); INV = json.load(open(P_INV))
TIERS = {k: SPEC[k] for k in ('Q_CTX', 'Q_CLIN', 'Q_FULL')}
LAB40 = INV['labels_task1']
TIER_OF = {l: t for t in ('head', 'medium', 'tail') for l in INV['tier'][t]}

E = pd.read_parquet(P_EHR)
# nhan o muc ANH; da kiem hang so trong study (28 C4) nen lay dai dien moi study
Y = (pd.read_parquet(P_IDX, columns=['study_id'] + LAB40)
       .drop_duplicates('study_id'))
DF = E.merge(Y, on='study_id', how='inner')
assert len(DF) == len(E), f'{len(E)-len(DF)} study cua EHR khong co nhan'
print(f'{len(DF):,} study | {len(LAB40)} nhan | split {DF.split.value_counts().to_dict()}')
print(f'tier: ' + ' | '.join(f'{k} {len(v)}' for k, v in TIERS.items()))

TR, VA = DF.split == 'train', DF.split == 'val'
print(f'train {TR.sum():,} | val {VA.sum():,}')
LABS = LAB40[:6] if QUICK else LAB40
if QUICK: print(f'\n!! QUICK: chi chay {len(LABS)} nhan')

Mounted at /content/drive
227,835 study | 40 nhan | split {'train': 155823, 'test': 48212, 'val': 23800}
tier: Q_CTX 157 | Q_CLIN 212 | Q_FULL 247
train 155,823 | val 23,800


## C1 · Ma trận đặc trưng cho từng tier

`gender` đã là 0/1 từ `29b`. Các cột còn lại đều là số. Giá trị thiếu **giữ NaN** — XGBoost xử lý
NaN nguyên bản bằng hướng đi mặc định ở mỗi nút, tốt hơn nhiều so với điền trung bình (điền sẽ
biến "không được đo" thành một giá trị lâm sàng giả).

Cell này cũng **kiểm lại `Q_CTX` không lọt giá trị lâm sàng** — nếu lọt thì cả ba phép so đều vô
nghĩa, nên kiểm ở đây chứ không tin vào `29b`.

In [ ]:
import re
VAL_RE = re.compile(r'^(lab_\w+_(24h|7d)$|vital_(temp_c|hr|rr|spo2|sbp|dbp)$|como_|age$|gender$)')
X = {}
for t, cols in TIERS.items():
    cols = [c for c in cols if c in DF.columns]
    if t == 'Q_CTX':
        leak = [c for c in cols if VAL_RE.match(c)]
        assert not leak, f'Q_CTX ro ri gia tri lam sang: {leak[:8]}'
    X[t] = DF[cols].apply(pd.to_numeric, errors='coerce').astype('float32').to_numpy()
    print(f'{t:<7} {X[t].shape[1]:>4} cot | NaN {np.isnan(X[t]).mean():6.1%}')
assert X['Q_CTX'].shape[1] < X['Q_CLIN'].shape[1] < X['Q_FULL'].shape[1]

Q_CTX    157 cot | NaN  17.7%
Q_CLIN   212 cot | NaN  24.1%
Q_FULL   247 cot | NaN  20.9%


## C2 · Khớp XGBoost — mỗi (tier, nhãn) một model

Khớp trên `train`, chấm trên `val`. `scale_pos_weight` để mặc định 1: ta chấm bằng **AP/AUROC**
chứ không bằng accuracy ở ngưỡng 0.5, nên cân bằng lại lớp không giúp mà chỉ làm lệch xác suất.

Ghi từng dòng vào CSV ngay khi xong, nên đứt giữa chừng chạy lại là tiếp tục.

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

P_CSV = os.path.join(RES_DIR, 'ehr_only_val.csv')
done = set()
if RESUME and os.path.exists(P_CSV):
    _d = pd.read_csv(P_CSV); done = set(zip(_d.tier, _d.label, _d.arm))
    print(f'resume: da co {len(done)} ket qua')

XGB = dict(n_estimators=400, max_depth=6, learning_rate=0.06, subsample=0.8,
           colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0,
           tree_method='hist', n_jobs=-1, random_state=SEED, eval_metric='logloss')

def fit_one(xtr, ytr, xva, yva):
    m = XGBClassifier(**XGB); m.fit(xtr, ytr, verbose=False)
    p = m.predict_proba(xva)[:, 1]
    return p, average_precision_score(yva, p), roc_auc_score(yva, p)

rows, t0, n_run = [], time.time(), 0
todo = [(t, l, a) for t in TIERS for l in LABS for a in ('real', 'shuf')
        if (t, l, a) not in done and (a == 'real' or t == 'Q_FULL')]
print(f'can khop: {len(todo)} model  ({len(LABS)} nhan x {len(TIERS)} tier + shuffle tren Q_FULL)')

rng = np.random.default_rng(SEED)
for k, (t, lab, arm) in enumerate(todo, 1):
    y = DF[lab].to_numpy('int8')
    ytr = y[TR.to_numpy()].copy()
    if arm == 'shuf': rng.shuffle(ytr)              # xao NHAN tren train, giu nguyen val
    if ytr.sum() < 5 or y[VA.to_numpy()].sum() < 1:
        print(f'  bo qua {t}/{lab}/{arm}: qua it ca duong'); continue
    p, ap, auc = fit_one(X[t][TR.to_numpy()], ytr, X[t][VA.to_numpy()], y[VA.to_numpy()])
    r = dict(tier=t, label=lab, arm=arm, ap=ap, auroc=auc,
             n_pos_train=int(y[TR.to_numpy()].sum()), n_pos_val=int(y[VA.to_numpy()].sum()),
             prevalence_val=float(y[VA.to_numpy()].mean()), tier_nhan=TIER_OF.get(lab, '?'))
    rows.append(r); n_run += 1
    pd.DataFrame([r]).to_csv(P_CSV, mode='a', index=False, header=not os.path.exists(P_CSV))
    if arm == 'real':
        np.save(os.path.join(RES_DIR, f'p_{t}_{lab.replace(" ","_")}.npy'), p.astype('float32'))
    el = time.time() - t0
    print(f'  {k:>3}/{len(todo)} {t:<7} {lab:<26} {arm:<4} AP {ap:.4f} AUROC {auc:.4f} '
          f'| {el/60:.1f}m, con ~{(len(todo)-k)*el/max(n_run,1)/60:.0f}m', end='\r')
print(f'\nxong {n_run} model trong {(time.time()-t0)/60:.1f} phut')

resume: da co 18 ket qua
can khop: 142 model  (40 nhan x 3 tier + shuffle tren Q_FULL)
  142/142 Q_FULL  Tuberculosis               shuf AP 0.0093 AUROC 0.5287 | 39.8m, con ~0m
xong 142 model trong 39.8 phut


## C3 · Bảng kết quả — kèm `n_pos`, tách theo tier nhãn

In [ ]:
R = pd.read_csv(P_CSV).drop_duplicates(['tier', 'label', 'arm'], keep='last')
REAL = R[R.arm == 'real']
piv = REAL.pivot(index='label', columns='tier', values='ap')
meta = REAL.drop_duplicates('label').set_index('label')[
    ['n_pos_train', 'n_pos_val', 'prevalence_val', 'tier_nhan']]
T = meta.join(piv).sort_values('n_pos_train', ascending=False)
T['CLIN-CTX'] = T.get('Q_CLIN', np.nan) - T.get('Q_CTX', np.nan)
T['FULL-CLIN'] = T.get('Q_FULL', np.nan) - T.get('Q_CLIN', np.nan)

print('=== AP tren VAL, tung nhan (sap theo so ca duong tren train) ===')
print(T.to_string(float_format=lambda v: f'{v:.4f}'))

print('\n=== MACRO theo TIER NHAN ===')
g = T.groupby('tier_nhan').agg(n_nhan=('n_pos_val', 'size'), n_pos_val_min=('n_pos_val', 'min'),
                               **{c: (c, 'mean') for c in ('Q_CTX', 'Q_CLIN', 'Q_FULL',
                                                           'CLIN-CTX', 'FULL-CLIN') if c in T})
print(g.reindex(['head', 'medium', 'tail']).to_string(float_format=lambda v: f'{v:.4f}'))

print('\n=== MACRO TOAN BO 40 NHAN ===')
for c in ('Q_CTX', 'Q_CLIN', 'Q_FULL'):
    if c in T: print(f'  {c:<7} mAP {T[c].mean():.4f} | mAUROC '
                     f'{REAL[REAL.tier==c].auroc.mean():.4f}')
print(f'\n  Q_CLIN - Q_CTX  = {T["CLIN-CTX"].mean():+.4f}   <- gia tri lam sang them duoc bao nhieu')
print(f'  Q_FULL - Q_CLIN = {T["FULL-CLIN"].mean():+.4f}   <- vi sinh them duoc bao nhieu')
print(f'''
DOC THE NAO:
  Q_CLIN - Q_CTX ~ 0  => EHR chi doc ra LOAI BENH NHAN va cuong do cham soc,
                         khong doc ra BENH. Do la ket luan quan trong nhat o day.
  Nhan tail (n_pos_val < 50) co phuong sai rat lon — dung dien giai chenh lech nho.''')

=== AP tren VAL, tung nhan (sap theo so ca duong tren train) ===
                            n_pos_train  n_pos_val  prevalence_val tier_nhan  Q_CLIN  Q_CTX  Q_FULL  CLIN-CTX  FULL-CLIN
label                                                                                                                   
Support Devices                   63040       9661          0.4059      head  0.7727 0.7584  0.7769    0.0143     0.0042
Lung Opacity                      49435       7539          0.3168      head  0.4430 0.3863  0.4497    0.0567     0.0067
Cardiomegaly                      47536       7457          0.3133      head  0.5018 0.4383  0.5013    0.0636    -0.0005
Pleural Effusion                  45671       7206          0.3028      head  0.6001 0.5499  0.6000    0.0502    -0.0001
Atelectasis                       44060       6863          0.2884      head  0.4452 0.4299  0.4477    0.0153     0.0025
Pneumonia                         27025       4143          0.1741      head  0.2548 0.2

## C4 · Tách theo `setting` — **đọc bảng này trước mọi kết luận**

Nhóm `outpatient` **không có vitals** (MIMIC-IV không có bảng vital ngoại trú). Nếu AP tụt hẳn khi
chấm **riêng trong từng nhóm** `setting`, nghĩa là phần lớn tín hiệu đến từ việc phân biệt nội trú
với ngoại trú — tức phân tầng bối cảnh, không phải chẩn đoán.

In [ ]:
va = DF[VA].reset_index(drop=True)
out = []
for t in TIERS:
    for lab in LABS:
        f = os.path.join(RES_DIR, f'p_{t}_{lab.replace(" ","_")}.npy')
        if not os.path.exists(f): continue
        p = np.load(f); y = va[lab].to_numpy('int8')
        for st in ('TAT CA', 'inpatient', 'ed', 'outpatient'):
            m = np.ones(len(va), bool) if st == 'TAT CA' else (va.setting == st).to_numpy()
            if y[m].sum() < 10: continue
            out.append(dict(tier=t, label=lab, setting=st, n=int(m.sum()),
                            n_pos=int(y[m].sum()), ap=average_precision_score(y[m], p[m]),
                            prev=float(y[m].mean())))
S = pd.DataFrame(out)
if len(S):
    print('=== mAP theo SETTING (chi nhan co >=10 ca duong trong nhom) ===')
    w = S.pivot_table(index='tier', columns='setting', values='ap', aggfunc='mean')
    n = S.pivot_table(index='tier', columns='setting', values='label', aggfunc='count')
    print(w.reindex(columns=['TAT CA', 'inpatient', 'ed', 'outpatient']).to_string(
        float_format=lambda v: f'{v:.4f}'))
    print('\nso nhan duoc tinh trong moi o:')
    print(n.reindex(columns=['TAT CA', 'inpatient', 'ed', 'outpatient']).to_string())
    print('\n=== so voi PREVALENCE cua chinh nhom do (AP cua bo doan ngau nhien) ===')
    pv = S.pivot_table(index='tier', columns='setting', values='prev', aggfunc='mean')
    lift = (w - pv).reindex(columns=['TAT CA', 'inpatient', 'ed', 'outpatient'])
    print(lift.to_string(float_format=lambda v: f'{v:+.4f}'))
    print('   (AP tru prevalence. Gan 0 = khong hon doan mu trong nhom do.)')
    S.to_csv(os.path.join(RES_DIR, 'ehr_only_by_setting.csv'), index=False)

=== mAP theo SETTING (chi nhan co >=10 ca duong trong nhom) ===
setting  TAT CA  inpatient     ed  outpatient
tier                                         
Q_CLIN   0.1295     0.1455 0.1158      0.1130
Q_CTX    0.1121     0.1296 0.0911      0.0938
Q_FULL   0.1313     0.1479 0.1182      0.1144

so nhan duoc tinh trong moi o:
setting  TAT CA  inpatient  ed  outpatient
tier                                      
Q_CLIN       40         36  35          35
Q_CTX        40         36  35          35
Q_FULL       40         36  35          35

=== so voi PREVALENCE cua chinh nhom do (AP cua bo doan ngau nhien) ===
setting  TAT CA  inpatient      ed  outpatient
tier                                          
Q_CLIN  +0.0632    +0.0565 +0.0579     +0.0484
Q_CTX   +0.0458    +0.0406 +0.0332     +0.0292
Q_FULL  +0.0651    +0.0589 +0.0603     +0.0498
   (AP tru prevalence. Gan 0 = khong hon doan mu trong nhom do.)


## C5 · Shuffle control + kết luận

In [ ]:
SH = R[(R.arm == 'shuf') & (R.tier == 'Q_FULL')]
if len(SH):
    j = (REAL[REAL.tier == 'Q_FULL'][['label', 'ap', 'prevalence_val', 'n_pos_val']]
         .merge(SH[['label', 'ap']], on='label', suffixes=('_that', '_xao')))
    print('=== SHUFFLE CONTROL tren Q_FULL ===')
    print(f'  mAP that : {j.ap_that.mean():.4f}')
    print(f'  mAP xao  : {j.ap_xao.mean():.4f}')
    print(f'  prevalence trung binh (san AP): {j.prevalence_val.mean():.4f}')
    _bad = j[j.ap_xao > j.prevalence_val * 2]
    if len(_bad):
        print(f'\n!! {len(_bad)} nhan co AP-xao > 2x prevalence -> NGHI RO RI:')
        print(_bad.nlargest(5, 'ap_xao')[['label', 'ap_xao', 'prevalence_val']].to_string(index=False))
    else:
        print('  OK  moi nhan xao deu ve sat prevalence -> khong co ro ri qua duong dac trung')

print(f'''
================================================================
STAGE 2 (EHR-only) XONG -> {RES_DIR}

  ehr_only_val.csv         AP/AUROC tung (tier, nhan, arm)
  ehr_only_by_setting.csv  AP tach theo setting
  p_<tier>_<nhan>.npy      xac suat tren val, de 32 gop voi CXR

TIEP THEO
  31_cxrlt_embed.ipynb  trich embedding RAD-DINO (GPU)
  32_cxrlt_baselines.ipynb  so EHR-only / CXR-only / EHR+CXR
                            va so ba arm anh (crop518 / lb518 / lb1022)

MOC DOI CHIEU (CXR-LT 2024, tu REVIEW_CXRLT_STAGE0.md §8):
  Task 1 vo dich 0.281 mAP | head 0.567 / medium 0.263 / tail 0.136
  Hang 2-4 cach nhau 0.004 -> chenh lech duoi 0.005 la nhieu.
  Do la diem cua mo hinh ANH fine-tune + ensemble; EHR-only se thap hon nhieu,
  va do la binh thuong — no do mot thu khac han.
================================================================''')

=== SHUFFLE CONTROL tren Q_FULL ===
  mAP that : 0.1313
  mAP xao  : 0.0693
  prevalence trung binh (san AP): 0.0663

!! 2 nhan co AP-xao > 2x prevalence -> NGHI RO RI:
        label   ap_xao  prevalence_val
Pleural Other 0.008740        0.001765
     Kyphosis 0.005993        0.002059

STAGE 2 (EHR-only) XONG -> /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/stage2_ehr_only

  ehr_only_val.csv         AP/AUROC tung (tier, nhan, arm)
  ehr_only_by_setting.csv  AP tach theo setting
  p_<tier>_<nhan>.npy      xac suat tren val, de 32 gop voi CXR

TIEP THEO
  31_cxrlt_embed.ipynb  trich embedding RAD-DINO (GPU)
  32_cxrlt_baselines.ipynb  so EHR-only / CXR-only / EHR+CXR
                            va so ba arm anh (crop518 / lb518 / lb1022)

MOC DOI CHIEU (CXR-LT 2024, tu REVIEW_CXRLT_STAGE0.md §8):
  Task 1 vo dich 0.281 mAP | head 0.567 / medium 0.263 / tail 0.136
  Hang 2-4 cach nhau 0.004 -> chenh lech duoi 0.005 la nhieu.
  Do la diem cua mo hinh ANH fine-tune + ensemble; EHR-